# Principled Abstention in Ternary Logic Gate Networks

In [ ]:
# === CELL 1: Imports + plot config ===
import os
os.environ["JAX_PLATFORMS"] = "cpu"

import time
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy.special import expit
from scipy.stats import pearsonr

from sklearn.datasets import make_moons, make_circles

# PST-DTLGN imports
from pst_dtlgn.data.pipeline import (
    TernaryPipeline, BinaryPipeline,
    ternarize_hard, ternarize_soft,
    uniform_thresholds, binarize,
)
from pst_dtlgn.network.topology import random_sparse
from pst_dtlgn.network.network import PolynomialNetwork
from pst_dtlgn.network.harden import harden_network, TernaryLearnedCircuit
from pst_dtlgn.core.gate_library import GateLibrary
from pst_dtlgn.training.trainer import train
from pst_dtlgn.binary_baseline import (
    BinaryDLGN, GroupSum, harden_binary_network, BinaryLearnedCircuit,
)
from pst_dtlgn.training.binary_trainer import train_binary
from pst_dtlgn.analysis.metrics import gate_diversity, functional_redundancy

# Publication-quality plot settings
sns.set_theme(style="whitegrid", font_scale=1.3)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.family": "serif",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 16,
})
os.makedirs("plots", exist_ok=True)

GATE_LIB = GateLibrary()
SEED = 42

print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
# === CELL 2: Dataset generators + train/test split ===

def make_spirals(n_samples=2000, noise=0.5, random_state=42):
    """Two interleaving spirals."""
    rng = np.random.RandomState(random_state)
    n = n_samples // 2
    theta = np.linspace(0, 3 * np.pi, n)
    r = theta / (3 * np.pi)
    x0 = np.column_stack([r * np.cos(theta), r * np.sin(theta)])
    x1 = np.column_stack([-r * np.cos(theta), -r * np.sin(theta)])
    X = np.vstack([x0, x1]).astype(np.float32)
    y = np.hstack([np.zeros(n), np.ones(n)]).astype(np.int32)
    X += rng.randn(*X.shape).astype(np.float32) * noise * 0.05
    return X, y

def make_gaussians(n_samples=2000, separation=1.5, random_state=42):
    """Two Gaussian blobs with controllable separation."""
    rng = np.random.RandomState(random_state)
    n = n_samples // 2
    X0 = rng.randn(n, 2).astype(np.float32) * 0.5 + np.array([-separation / 2, 0])
    X1 = rng.randn(n, 2).astype(np.float32) * 0.5 + np.array([separation / 2, 0])
    X = np.vstack([X0, X1])
    y = np.hstack([np.zeros(n), np.ones(n)]).astype(np.int32)
    return X, y

def make_ring_sector(n_samples=2000, random_state=42):
    """Ring sector: inner circle (class 0) vs outer ring sector (class 1)."""
    rng = np.random.RandomState(random_state)
    n = n_samples // 2
    # Class 0: inner circle
    r0 = rng.uniform(0, 0.4, n)
    theta0 = rng.uniform(0, 2 * np.pi, n)
    x0 = np.column_stack([r0 * np.cos(theta0), r0 * np.sin(theta0)])
    # Class 1: outer ring
    r1 = rng.uniform(0.7, 1.2, n)
    theta1 = rng.uniform(0, 2 * np.pi, n)
    x1 = np.column_stack([r1 * np.cos(theta1), r1 * np.sin(theta1)])
    X = np.vstack([x0, x1]).astype(np.float32)
    y = np.hstack([np.zeros(n), np.ones(n)]).astype(np.int32)
    idx = rng.permutation(len(X))
    return X[idx], y[idx]

def split_train_test(X, y, test_frac=0.2, random_state=42):
    """80/20 train/test split."""
    rng = np.random.RandomState(random_state)
    N = len(X)
    idx = rng.permutation(N)
    split = int(N * (1 - test_frac))
    return X[idx[:split]], y[idx[:split]], X[idx[split:]], y[idx[split:]]

# Generate all 5 datasets
DATASETS = {}
DATASETS_TEST = {}

# Moons
X, y = make_moons(n_samples=2000, noise=0.2, random_state=SEED)
X, y = X.astype(np.float32), y.astype(np.int32)
X_tr, y_tr, X_te, y_te = split_train_test(X, y)
DATASETS["moons"] = (X_tr, y_tr)
DATASETS_TEST["moons"] = (X_te, y_te)

# Circles
X, y = make_circles(n_samples=2000, noise=0.1, factor=0.5, random_state=SEED)
X, y = X.astype(np.float32), y.astype(np.int32)
X_tr, y_tr, X_te, y_te = split_train_test(X, y)
DATASETS["circles"] = (X_tr, y_tr)
DATASETS_TEST["circles"] = (X_te, y_te)

# Spirals
X, y = make_spirals(n_samples=2000, noise=0.5, random_state=SEED)
X_tr, y_tr, X_te, y_te = split_train_test(X, y)
DATASETS["spirals"] = (X_tr, y_tr)
DATASETS_TEST["spirals"] = (X_te, y_te)

# Gaussians (default separation=1.5)
X, y = make_gaussians(n_samples=2000, separation=1.5, random_state=SEED)
X_tr, y_tr, X_te, y_te = split_train_test(X, y)
DATASETS["gaussians"] = (X_tr, y_tr)
DATASETS_TEST["gaussians"] = (X_te, y_te)

# Ring sector
X, y = make_ring_sector(n_samples=2000, random_state=SEED)
X_tr, y_tr, X_te, y_te = split_train_test(X, y)
DATASETS["ring_sector"] = (X_tr, y_tr)
DATASETS_TEST["ring_sector"] = (X_te, y_te)

for name in DATASETS:
    Xtr, ytr = DATASETS[name]
    Xte, yte = DATASETS_TEST[name]
    print(f"{name:15s}  train={Xtr.shape[0]:5d}  test={Xte.shape[0]:4d}  dim={Xtr.shape[1]}")

ALL_DATASET_NAMES = ["moons", "circles", "spirals", "gaussians", "ring_sector"]

In [ ]:
# === CELL 3: Utility functions ===

# --- Architecture configs ---
ARCH_CONFIGS = {
    "tiny_single":   {"body": [64, 64, 64],      "output": 1,    "npc": None},
    "small_gs10":    {"body": [128, 128, 128],    "output": 20,   "npc": 10},
    "small_gs50":    {"body": [256, 256, 256],    "output": 100,  "npc": 50},
    "medium_gs100":  {"body": [512, 512, 512],    "output": 200,  "npc": 100},
    "large_gs200":   {"body": [1024, 1024, 1024], "output": 400,  "npc": 200},
}

def get_layer_widths(config_name):
    cfg = ARCH_CONFIGS[config_name]
    return cfg["body"] + [cfg["output"]]


# --- Loss wrappers ---
def make_ternary_gs_loss(group_sum):
    """GroupSum cross-entropy loss for ternary networks."""
    def loss_fn(preds, targets):
        logits = jax.vmap(group_sum)(preds)
        labels = targets.astype(jnp.int32).ravel()
        return jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, labels))
    return loss_fn


# --- Evaluation ---
def evaluate_ternary_circuit_gs(circuit, X_encoded, k, npc):
    """Evaluate hardened ternary circuit with GroupSum aggregation."""
    N = X_encoded.shape[0]
    preds = np.zeros(N, dtype=np.int32)
    margins = np.zeros(N, dtype=np.float64)
    total_zeros = 0
    total_outputs = 0

    for i in range(N):
        out = circuit(X_encoded[i])
        out_float = out.astype(np.float64)
        total_zeros += np.sum(out == 0)
        total_outputs += len(out)
        scores = out_float.reshape(k, npc).sum(axis=-1)
        sorted_scores = np.sort(scores)[::-1]
        preds[i] = np.argmax(scores)
        margins[i] = sorted_scores[0] - sorted_scores[1]

    unknown_frac = total_zeros / max(total_outputs, 1)
    return preds, margins, unknown_frac


def evaluate_binary_circuit_gs(circuit, X_encoded, k, npc):
    """Evaluate hardened binary circuit with GroupSum aggregation."""
    N = X_encoded.shape[0]
    preds = np.zeros(N, dtype=np.int32)
    margins = np.zeros(N, dtype=np.float64)

    for i in range(N):
        out = circuit(X_encoded[i])
        out_float = out.astype(np.float64)
        scores = out_float.reshape(k, npc).sum(axis=-1)
        sorted_scores = np.sort(scores)[::-1]
        preds[i] = np.argmax(scores)
        margins[i] = sorted_scores[0] - sorted_scores[1]

    return preds, margins


def evaluate_soft_model_gs(model, X_encoded, group_sum):
    """Evaluate soft (pre-hardening) model with GroupSum."""
    X_jax = jnp.asarray(X_encoded)
    logits = jax.vmap(lambda x: group_sum(model(x)))(X_jax)
    probs = jax.nn.softmax(logits, axis=-1)
    preds = np.asarray(jnp.argmax(logits, axis=-1))
    return preds, np.asarray(probs)


# --- Accuracy vs coverage ---
def accuracy_vs_coverage(preds, labels, margins, n_thresholds=200):
    """Compute accuracy-vs-coverage curve (selective prediction)."""
    thresholds = np.linspace(0, np.max(margins) + 1e-6, n_thresholds)
    coverages = np.zeros(n_thresholds)
    accuracies = np.zeros(n_thresholds)
    for i, t in enumerate(thresholds):
        decided = margins >= t
        n_decided = np.sum(decided)
        coverages[i] = n_decided / len(preds)
        if n_decided > 0:
            accuracies[i] = np.mean(preds[decided] == labels[decided])
        else:
            accuracies[i] = np.nan
    return coverages, accuracies, thresholds


# --- Decision grid ---
def make_decision_grid(X_raw, pipeline, circuit, k, npc, grid_res=200):
    """Compute decision boundary grid for a 2D dataset."""
    x_min, x_max = X_raw[:, 0].min() - 0.5, X_raw[:, 0].max() + 0.5
    y_min, y_max = X_raw[:, 1].min() - 0.5, X_raw[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, grid_res),
        np.linspace(y_min, y_max, grid_res),
    )
    grid_points = np.column_stack([xx.ravel(), yy.ravel()]).astype(np.float32)

    is_ternary = isinstance(pipeline, TernaryPipeline) or hasattr(pipeline, 'thresholds')
    encoded = pipeline.transform(grid_points, mode="hard")

    if is_ternary:
        preds, margins, _ = evaluate_ternary_circuit_gs(circuit, encoded, k, npc)
        grid_unknowns = np.zeros(len(grid_points))
        for i in range(len(grid_points)):
            out = circuit(encoded[i])
            grid_unknowns[i] = np.mean(out == 0)
        grid_unknowns = grid_unknowns.reshape(grid_res, grid_res)
    else:
        preds, margins = evaluate_binary_circuit_gs(circuit, encoded, k, npc)
        grid_unknowns = np.zeros((grid_res, grid_res))

    return xx, yy, preds.reshape(grid_res, grid_res), margins.reshape(grid_res, grid_res), grid_unknowns


# --- Asymmetric encoding ---
def ternarize_hard_asymmetric(features, thresholds, delta_fraction=1.0):
    """Ternary encoding with tunable UNKNOWN band width."""
    N, D = features.shape
    K = len(thresholds)
    sorted_t = np.sort(thresholds)
    if K == 1:
        auto_margin = 0.1
    else:
        auto_margin = np.min(np.diff(sorted_t)) / 2.0
    margin = delta_fraction * auto_margin
    f = features[:, :, None]
    t = sorted_t[None, None, :]
    if margin < 1e-10:
        result = np.where(f > t, 1.0, -1.0)
    else:
        result = np.where(f > t + margin, 1.0,
                 np.where(f < t - margin, -1.0, 0.0))
    return result.reshape(N, D * K).astype(np.float32)


class AsymmetricTernaryPipeline:
    """TernaryPipeline with tunable delta_fraction for UNKNOWN band width."""
    def __init__(self, resolution=4, delta_fraction=1.0):
        self.thresholds = uniform_thresholds(resolution)
        self.delta_fraction = delta_fraction
        self.feature_min = None
        self.feature_max = None
        self.input_dim = None

    def fit(self, features):
        self.feature_min = np.min(features, axis=0)
        self.feature_max = np.max(features, axis=0)
        D = features.shape[1]
        K = len(self.thresholds)
        self.input_dim = D * K
        return self

    def transform(self, features, mode="hard"):
        assert self.feature_min is not None, "Call fit() first"
        denom = self.feature_max - self.feature_min
        denom = np.where(denom < 1e-8, 1.0, denom)
        normalized = (features - self.feature_min) / denom
        normalized = np.clip(normalized, 0.0, 1.0)
        return ternarize_hard_asymmetric(normalized, self.thresholds, self.delta_fraction)

    def fit_transform(self, features, mode="hard"):
        return self.fit(features).transform(features, mode=mode)


# --- Training wrappers ---
def train_ternary_gs(dataset_name, layer_widths, npc, resolution=4,
                     lr=0.003, lambda_max=0.1, lambda_gamma=2.0,
                     steps=5000, batch_size=64, seed=SEED, log_every=500,
                     pipeline=None):
    """Train a ternary PST-DTLGN with GroupSum output."""
    X_raw, y = DATASETS[dataset_name]
    k = 2

    if pipeline is None:
        pipe = TernaryPipeline(resolution=resolution)
    else:
        pipe = pipeline
    X_enc = pipe.fit_transform(X_raw, mode="hard")
    input_dim = pipe.input_dim

    X_jax = jnp.asarray(X_enc)
    y_jax = jnp.asarray(y)

    key = jax.random.PRNGKey(seed)
    key, topo_key = jax.random.split(key)
    topo = random_sparse(topo_key, input_dim, layer_widths)
    net = PolynomialNetwork(key, topo)

    gs = GroupSum(k=k, tau=10.0)
    loss_fn = make_ternary_gs_loss(gs)

    optimizer = optax.adam(lr)
    t0 = time.time()
    trained, history = train(
        net, optimizer, X_jax, y_jax,
        total_steps=steps, lambda_max=lambda_max, lambda_gamma=lambda_gamma,
        batch_size=batch_size, loss_fn=loss_fn, key=key, log_every=log_every,
    )
    train_time = time.time() - t0

    harden_result = harden_network(trained, GATE_LIB)
    circuit = TernaryLearnedCircuit(harden_result)

    soft_preds, soft_probs = evaluate_soft_model_gs(trained, X_enc, gs)
    soft_acc = np.mean(soft_preds == y)

    hard_preds, hard_margins, unk_frac = evaluate_ternary_circuit_gs(circuit, X_enc, k, npc)
    hard_acc = np.mean(hard_preds == y)
    gap = soft_acc - hard_acc

    # Test set evaluation
    X_te, y_te = DATASETS_TEST[dataset_name]
    X_te_enc = pipe.transform(X_te, mode="hard")
    te_preds, te_margins, te_unk = evaluate_ternary_circuit_gs(circuit, X_te_enc, k, npc)
    test_acc = np.mean(te_preds == y_te)

    return {
        "model": trained, "circuit": circuit, "pipeline": pipe,
        "harden_result": harden_result, "history": history,
        "group_sum": gs, "k": k, "npc": npc,
        "soft_acc": soft_acc, "hard_acc": hard_acc, "gap": gap,
        "unknown_frac": unk_frac, "train_time": train_time,
        "test_acc": test_acc, "test_unk": te_unk,
        "test_preds": te_preds, "test_margins": te_margins,
        "train_preds": hard_preds, "train_margins": hard_margins,
        "config": {
            "dataset": dataset_name, "layer_widths": layer_widths,
            "npc": npc, "resolution": resolution, "lr": lr,
            "lambda_max": lambda_max, "steps": steps,
        },
    }


def train_binary_gs(dataset_name, layer_widths, npc, resolution=4,
                    lr=0.003, steps=5000, batch_size=64, seed=SEED):
    """Train a binary DLGN with GroupSum output."""
    X_raw, y = DATASETS[dataset_name]
    k = 2

    pipe = BinaryPipeline(resolution=resolution)
    X_enc = pipe.fit_transform(X_raw, mode="hard")
    input_dim = pipe.input_dim

    X_jax = jnp.asarray(X_enc)
    y_jax = jnp.asarray(y)

    key = jax.random.PRNGKey(seed)
    key, topo_key = jax.random.split(key)
    topo = random_sparse(topo_key, input_dim, layer_widths)
    gs = GroupSum(k=k, tau=10.0)
    net = BinaryDLGN(key, topo, init="randn")

    optimizer = optax.adam(lr)
    t0 = time.time()
    trained, history = train_binary(
        net, optimizer, X_jax, y_jax,
        total_steps=steps, batch_size=batch_size,
        group_sum=gs, key=key, log_every=500,
    )
    train_time = time.time() - t0

    harden_result = harden_binary_network(trained)
    circuit = BinaryLearnedCircuit(harden_result)

    soft_preds, soft_probs = evaluate_soft_model_gs(trained, X_enc, gs)
    soft_acc = np.mean(soft_preds == y)

    hard_preds, hard_margins = evaluate_binary_circuit_gs(circuit, X_enc, k, npc)
    hard_acc = np.mean(hard_preds == y)
    gap = soft_acc - hard_acc

    # Test set evaluation
    X_te, y_te = DATASETS_TEST[dataset_name]
    X_te_enc = pipe.transform(X_te, mode="hard")
    te_preds, te_margins = evaluate_binary_circuit_gs(circuit, X_te_enc, k, npc)
    test_acc = np.mean(te_preds == y_te)

    return {
        "model": trained, "circuit": circuit, "pipeline": pipe,
        "harden_result": harden_result, "history": history,
        "group_sum": gs, "k": k, "npc": npc,
        "soft_acc": soft_acc, "hard_acc": hard_acc, "gap": gap,
        "unknown_frac": 0.0, "train_time": train_time,
        "test_acc": test_acc, "test_unk": 0.0,
        "test_preds": te_preds, "test_margins": te_margins,
        "train_preds": hard_preds, "train_margins": hard_margins,
        "config": {
            "dataset": dataset_name, "layer_widths": layer_widths,
            "npc": npc, "resolution": resolution, "lr": lr, "steps": steps,
        },
    }


print("All utility functions defined.")

In [ ]:
# === CELL 4: Train all models ===
# Architecture: medium_gs100 -> [512, 512, 512, 200], npc=100

CONFIG_NAME = "medium_gs100"
LAYER_WIDTHS = get_layer_widths(CONFIG_NAME)
NPC = ARCH_CONFIGS[CONFIG_NAME]["npc"]

print(f"Config: {CONFIG_NAME}")
print(f"Layer widths: {LAYER_WIDTHS}")
print(f"NPC: {NPC}")
print(f"Resolution: 4 (K=3, input_dim=6 for 2D data)")
print()

RESULTS = {}

for ds_name in ALL_DATASET_NAMES:
    print(f"\n{'='*60}")
    print(f"Training on: {ds_name}")
    print(f"{'='*60}")

    # Ternary
    print(f"  [Ternary] Training...")
    t_result = train_ternary_gs(
        ds_name, LAYER_WIDTHS, NPC,
        resolution=4, lr=0.003, lambda_max=0.1, lambda_gamma=2.0,
        steps=5000, batch_size=64,
    )
    print(f"  [Ternary] Train acc: {t_result['hard_acc']:.3f}  "
          f"Test acc: {t_result['test_acc']:.3f}  "
          f"Gap: {t_result['gap']:.4f}  "
          f"UNK: {t_result['unknown_frac']:.3f}  "
          f"Time: {t_result['train_time']:.1f}s")

    # Binary
    print(f"  [Binary]  Training...")
    b_result = train_binary_gs(
        ds_name, LAYER_WIDTHS, NPC,
        resolution=4, lr=0.003, steps=5000, batch_size=64,
    )
    print(f"  [Binary]  Train acc: {b_result['hard_acc']:.3f}  "
          f"Test acc: {b_result['test_acc']:.3f}  "
          f"Gap: {b_result['gap']:.4f}  "
          f"Time: {b_result['train_time']:.1f}s")

    RESULTS[ds_name] = {"ternary": t_result, "binary": b_result}

print("\n" + "="*60)
print("Training complete for all datasets.")
print("="*60)

In [ ]:
# === CELL 5: Cross-dataset summary table ===

print(f"{'Dataset':15s} | {'Tern Train':>10s} {'Tern Test':>10s} {'UNK%':>6s} {'Gap':>8s} | "
      f"{'Bin Train':>10s} {'Bin Test':>10s} {'Gap':>8s}")
print("-" * 95)

for ds_name in ALL_DATASET_NAMES:
    t = RESULTS[ds_name]["ternary"]
    b = RESULTS[ds_name]["binary"]
    print(f"{ds_name:15s} | "
          f"{t['hard_acc']:10.3f} {t['test_acc']:10.3f} {t['unknown_frac']*100:5.1f}% {t['gap']:8.4f} | "
          f"{b['hard_acc']:10.3f} {b['test_acc']:10.3f} {b['gap']:8.4f}")

## Figure 1: Decision Boundary Gallery

In [ ]:
# === CELL 7: Compute decision grids for all 10 circuits ===

GRIDS = {}
GRID_RES = 200

for ds_name in ALL_DATASET_NAMES:
    X_raw = np.vstack([DATASETS[ds_name][0], DATASETS_TEST[ds_name][0]])

    # Ternary grid
    t_res = RESULTS[ds_name]["ternary"]
    t_grid = make_decision_grid(
        X_raw, t_res["pipeline"], t_res["circuit"],
        t_res["k"], t_res["npc"], grid_res=GRID_RES,
    )

    # Binary grid
    b_res = RESULTS[ds_name]["binary"]
    b_grid = make_decision_grid(
        X_raw, b_res["pipeline"], b_res["circuit"],
        b_res["k"], b_res["npc"], grid_res=GRID_RES,
    )

    GRIDS[ds_name] = {"ternary": t_grid, "binary": b_grid}
    print(f"{ds_name}: grids computed")

print("All grids computed.")

In [ ]:
# === CELL 8: Figure 1 -- 5x4 decision boundary gallery ===

fig, axes = plt.subplots(5, 4, figsize=(20, 25))

DATASET_LABELS = {
    "moons": "Moons",
    "circles": "Circles",
    "spirals": "Spirals",
    "gaussians": "Gaussians",
    "ring_sector": "Ring Sector",
}

for row_idx, ds_name in enumerate(ALL_DATASET_NAMES):
    X_all = np.vstack([DATASETS[ds_name][0], DATASETS_TEST[ds_name][0]])
    y_all = np.hstack([DATASETS[ds_name][1], DATASETS_TEST[ds_name][1]])

    t_grid = GRIDS[ds_name]["ternary"]
    b_grid = GRIDS[ds_name]["binary"]
    xx, yy = t_grid[0], t_grid[1]

    # Column 0: Data scatter
    ax = axes[row_idx, 0]
    colors = ['#3274A1', '#E1812C']
    for c in [0, 1]:
        mask = y_all == c
        ax.scatter(X_all[mask, 0], X_all[mask, 1], c=colors[c],
                   s=4, alpha=0.5, rasterized=True)
    ax.set_title(f"{DATASET_LABELS[ds_name]}" if row_idx == 0 else "", fontsize=13)
    ax.set_ylabel(DATASET_LABELS[ds_name], fontsize=13, fontweight='bold')
    if row_idx == 0:
        ax.set_title("Data", fontsize=13, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

    # Column 1: Binary boundary
    ax = axes[row_idx, 1]
    ax.contourf(xx, yy, b_grid[2], levels=[-0.5, 0.5, 1.5],
                colors=['#a6c8e0', '#f5c4a1'], alpha=0.7)
    ax.contour(xx, yy, b_grid[2], levels=[0.5], colors='black', linewidths=1.5)
    for c in [0, 1]:
        mask = y_all == c
        ax.scatter(X_all[mask, 0], X_all[mask, 1], c=colors[c],
                   s=3, alpha=0.3, rasterized=True)
    b_acc = RESULTS[ds_name]["binary"]["test_acc"]
    if row_idx == 0:
        ax.set_title("Binary DLGN", fontsize=13, fontweight='bold')
    ax.text(0.02, 0.98, f"Acc: {b_acc:.1%}", transform=ax.transAxes,
            va='top', ha='left', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    ax.set_xticks([])
    ax.set_yticks([])

    # Column 2: Ternary boundary
    ax = axes[row_idx, 2]
    ax.contourf(xx, yy, t_grid[2], levels=[-0.5, 0.5, 1.5],
                colors=['#a6c8e0', '#f5c4a1'], alpha=0.7)
    ax.contour(xx, yy, t_grid[2], levels=[0.5], colors='black', linewidths=1.5)
    for c in [0, 1]:
        mask = y_all == c
        ax.scatter(X_all[mask, 0], X_all[mask, 1], c=colors[c],
                   s=3, alpha=0.3, rasterized=True)
    t_acc = RESULTS[ds_name]["ternary"]["test_acc"]
    if row_idx == 0:
        ax.set_title("Ternary PST-DTLGN", fontsize=13, fontweight='bold')
    ax.text(0.02, 0.98, f"Acc: {t_acc:.1%}", transform=ax.transAxes,
            va='top', ha='left', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    ax.set_xticks([])
    ax.set_yticks([])

    # Column 3: UNKNOWN overlay
    ax = axes[row_idx, 3]
    # Background: ternary decision
    ax.contourf(xx, yy, t_grid[2], levels=[-0.5, 0.5, 1.5],
                colors=['#d4e6f1', '#fae5d3'], alpha=0.3)
    # UNKNOWN heatmap overlay
    unk = t_grid[4]
    im = ax.imshow(unk, extent=[xx.min(), xx.max(), yy.min(), yy.max()],
                   origin='lower', cmap='Oranges', alpha=0.8,
                   vmin=0, vmax=max(unk.max(), 0.01), aspect='auto')
    unk_pct = RESULTS[ds_name]["ternary"]["unknown_frac"]
    if row_idx == 0:
        ax.set_title("UNKNOWN Density", fontsize=13, fontweight='bold')
    ax.text(0.02, 0.98, f"UNK: {unk_pct:.1%}", transform=ax.transAxes,
            va='top', ha='left', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout(h_pad=0.5, w_pad=0.5)
plt.savefig("plots/FINAL_decision_boundaries.svg", bbox_inches="tight")
plt.savefig("plots/FINAL_decision_boundaries.pdf", bbox_inches="tight")
plt.show()
print("Saved: plots/FINAL_decision_boundaries.svg, .pdf")

## Figure 2: Accuracy vs Coverage

In [ ]:
# === CELL 10: Figure 2 -- accuracy vs coverage (3 highlight datasets) ===

HIGHLIGHT_DATASETS = ["spirals", "moons", "gaussians"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, ds_name in enumerate(HIGHLIGHT_DATASETS):
    ax = axes[idx]
    X_te, y_te = DATASETS_TEST[ds_name]

    # Ternary
    t_res = RESULTS[ds_name]["ternary"]
    t_preds = t_res["test_preds"]
    t_margins = t_res["test_margins"]
    t_cov, t_acc, _ = accuracy_vs_coverage(t_preds, y_te, t_margins)

    # Binary
    b_res = RESULTS[ds_name]["binary"]
    b_preds = b_res["test_preds"]
    b_margins = b_res["test_margins"]
    b_cov, b_acc, _ = accuracy_vs_coverage(b_preds, y_te, b_margins)

    # Plot ternary curve
    valid_t = ~np.isnan(t_acc)
    ax.plot(t_cov[valid_t], t_acc[valid_t], color='#E24A33', linewidth=2.5,
            label='Ternary PST-DTLGN', zorder=3)

    # Binary horizontal reference (full-coverage accuracy)
    b_full_acc = b_res["test_acc"]
    ax.axhline(b_full_acc, color='#348ABD', linewidth=2, linestyle='--',
               label=f'Binary DLGN ({b_full_acc:.1%})', zorder=2)

    # Shade area between curves
    ax.fill_between(t_cov[valid_t], b_full_acc, t_acc[valid_t],
                    where=t_acc[valid_t] >= b_full_acc,
                    alpha=0.15, color='#E24A33', zorder=1)

    # AUC annotation
    auc_t = np.trapezoid(t_acc[valid_t], t_cov[valid_t])
    ax.text(0.03, 0.05, f"AUC = {auc_t:.3f}", transform=ax.transAxes,
            fontsize=11, va='bottom',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

    ax.set_xlabel("Coverage", fontsize=12)
    ax.set_ylabel("Accuracy" if idx == 0 else "", fontsize=12)
    ax.set_title(DATASET_LABELS[ds_name], fontsize=14, fontweight='bold')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(0.45, 1.02)
    ax.legend(loc='lower left', fontsize=10)

plt.tight_layout()
plt.savefig("plots/FINAL_accuracy_vs_coverage.svg", bbox_inches="tight")
plt.savefig("plots/FINAL_accuracy_vs_coverage.pdf", bbox_inches="tight")
plt.show()
print("Saved: plots/FINAL_accuracy_vs_coverage.svg, .pdf")

## Figure 3: UNKNOWN and Bayes Uncertainty

In [ ]:
# === CELL 12: Bayes posterior + Gaussian separation sweep ===

# (a) Bayes posterior for Gaussians at sep=1.5
def bayes_posterior_gaussians(X, sep, sigma=0.5):
    """P(y=1|x) for two equal-prior isotropic Gaussians."""
    mu0 = np.array([-sep / 2, 0])
    mu1 = np.array([sep / 2, 0])
    # log p(x|y=0) - log p(x|y=1)
    log_ratio = (-0.5 / sigma**2) * (
        np.sum((X - mu0)**2, axis=-1) - np.sum((X - mu1)**2, axis=-1)
    )
    p1 = expit(-log_ratio)  # P(y=1|x) = sigmoid(log p1 - log p0)
    return p1

def bayes_uncertainty(p1):
    """Binary entropy H(p) = -p log p - (1-p) log (1-p)."""
    p0 = 1 - p1
    eps = 1e-12
    return -(p1 * np.log2(p0 + eps) + p0 * np.log2(p0 + eps))  # Intentional: H(Y|x)

def bayes_entropy(p1):
    """Binary entropy."""
    eps = 1e-12
    return -(p1 * np.log2(p1 + eps) + (1 - p1) * np.log2(1 - p1 + eps))

# (b) Separation sweep
SEPARATIONS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
sweep_results = []

for sep in SEPARATIONS:
    print(f"\nSeparation = {sep:.1f}")
    # Generate Gaussian dataset at this separation
    X_sweep, y_sweep = make_gaussians(n_samples=2000, separation=sep, random_state=SEED)
    X_tr_s, y_tr_s, X_te_s, y_te_s = split_train_test(X_sweep, y_sweep)

    # Temporarily add to DATASETS for training
    DATASETS["_gauss_sweep"] = (X_tr_s, y_tr_s)
    DATASETS_TEST["_gauss_sweep"] = (X_te_s, y_te_s)

    # Train ternary
    t_res = train_ternary_gs(
        "_gauss_sweep", LAYER_WIDTHS, NPC,
        resolution=4, lr=0.003, lambda_max=0.1, lambda_gamma=2.0,
        steps=5000, batch_size=64,
    )
    # Train binary
    b_res = train_binary_gs(
        "_gauss_sweep", LAYER_WIDTHS, NPC,
        resolution=4, lr=0.003, steps=5000, batch_size=64,
    )

    # Bayes accuracy on test set
    p1_test = bayes_posterior_gaussians(X_te_s, sep, sigma=0.5)
    bayes_preds = (p1_test >= 0.5).astype(np.int32)
    bayes_acc = np.mean(bayes_preds == y_te_s)

    sweep_results.append({
        "sep": sep,
        "ternary_acc": t_res["test_acc"],
        "binary_acc": b_res["test_acc"],
        "bayes_acc": bayes_acc,
        "ternary_unk": t_res["test_unk"],
        "ternary_result": t_res,
        "binary_result": b_res,
    })
    print(f"  Ternary: {t_res['test_acc']:.3f} (UNK={t_res['test_unk']:.3f})  "
          f"Binary: {b_res['test_acc']:.3f}  Bayes: {bayes_acc:.3f}")

# Clean up temp dataset
del DATASETS["_gauss_sweep"]
del DATASETS_TEST["_gauss_sweep"]

print("\nSweep complete.")

In [ ]:
# === CELL 13: Figure 3 -- 2x3 panel ===

fig = plt.figure(figsize=(20, 12))
gs_fig = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# --- Row 1, Panel (a): Bayes uncertainty heatmap at sep=1.5 ---
ax_a = fig.add_subplot(gs_fig[0, 0])
x_range = np.linspace(-3, 3, 200)
y_range = np.linspace(-3, 3, 200)
xx_b, yy_b = np.meshgrid(x_range, y_range)
grid_b = np.column_stack([xx_b.ravel(), yy_b.ravel()])
p1_grid = bayes_posterior_gaussians(grid_b, sep=1.5, sigma=0.5)
H_grid = bayes_entropy(p1_grid).reshape(200, 200)
im_a = ax_a.imshow(H_grid, extent=[-3, 3, -3, 3], origin='lower',
                   cmap='inferno', aspect='auto', vmin=0, vmax=1)
ax_a.set_title("(a) Bayes Uncertainty $H(Y|x)$", fontsize=13, fontweight='bold')
ax_a.set_xlabel("$x_1$")
ax_a.set_ylabel("$x_2$")
plt.colorbar(im_a, ax=ax_a, fraction=0.046, pad=0.04)

# --- Row 1, Panel (b): Ternary UNKNOWN overlay at sep=1.5 ---
ax_b = fig.add_subplot(gs_fig[0, 1])
# Use the sep=1.5 result from the main RESULTS (gaussians dataset)
gauss_t = RESULTS["gaussians"]["ternary"]
gauss_grid = GRIDS["gaussians"]["ternary"]
xx_g, yy_g = gauss_grid[0], gauss_grid[1]
unk_g = gauss_grid[4]
ax_b.contourf(xx_g, yy_g, gauss_grid[2], levels=[-0.5, 0.5, 1.5],
              colors=['#d4e6f1', '#fae5d3'], alpha=0.3)
im_b = ax_b.imshow(unk_g, extent=[xx_g.min(), xx_g.max(), yy_g.min(), yy_g.max()],
                   origin='lower', cmap='Oranges', alpha=0.8,
                   vmin=0, vmax=max(unk_g.max(), 0.01), aspect='auto')
ax_b.set_title("(b) Ternary UNKNOWN Density", fontsize=13, fontweight='bold')
ax_b.set_xlabel("$x_1$")
ax_b.set_ylabel("$x_2$")
plt.colorbar(im_b, ax=ax_b, fraction=0.046, pad=0.04)

# --- Row 1, Panel (c): Binary decision boundary at sep=1.5 ---
ax_c = fig.add_subplot(gs_fig[0, 2])
gauss_b = GRIDS["gaussians"]["binary"]
xx_gb, yy_gb = gauss_b[0], gauss_b[1]
ax_c.contourf(xx_gb, yy_gb, gauss_b[2], levels=[-0.5, 0.5, 1.5],
              colors=['#a6c8e0', '#f5c4a1'], alpha=0.7)
ax_c.contour(xx_gb, yy_gb, gauss_b[2], levels=[0.5], colors='black', linewidths=1.5)
X_all_g = np.vstack([DATASETS["gaussians"][0], DATASETS_TEST["gaussians"][0]])
y_all_g = np.hstack([DATASETS["gaussians"][1], DATASETS_TEST["gaussians"][1]])
for c in [0, 1]:
    mask = y_all_g == c
    ax_c.scatter(X_all_g[mask, 0], X_all_g[mask, 1],
                 c=['#3274A1', '#E1812C'][c], s=4, alpha=0.3, rasterized=True)
ax_c.set_title("(c) Binary Decision Boundary", fontsize=13, fontweight='bold')
ax_c.set_xlabel("$x_1$")
ax_c.set_ylabel("$x_2$")

# --- Row 2, Panel (d): Separation vs UNKNOWN% ---
ax_d = fig.add_subplot(gs_fig[1, 0])
seps = [r["sep"] for r in sweep_results]
unks = [r["ternary_unk"] * 100 for r in sweep_results]
ax_d.plot(seps, unks, 'o-', color='#E24A33', linewidth=2.5, markersize=8, label='UNKNOWN %')
ax_d.set_xlabel("Class Separation", fontsize=12)
ax_d.set_ylabel("UNKNOWN Output Fraction (%)", fontsize=12)
ax_d.set_title("(d) Separation vs UNKNOWN %", fontsize=13, fontweight='bold')
# Correlation annotation
if len(seps) > 2:
    r_corr, p_val = pearsonr(seps, unks)
    ax_d.text(0.95, 0.95, f"r = {r_corr:.2f}\np = {p_val:.3f}",
             transform=ax_d.transAxes, va='top', ha='right', fontsize=11,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

# --- Row 2, Panel (e): Separation vs accuracy (ternary + binary + Bayes) ---
ax_e = fig.add_subplot(gs_fig[1, 1])
t_accs = [r["ternary_acc"] * 100 for r in sweep_results]
b_accs = [r["binary_acc"] * 100 for r in sweep_results]
bayes_accs = [r["bayes_acc"] * 100 for r in sweep_results]
ax_e.plot(seps, t_accs, 'o-', color='#E24A33', linewidth=2.5, markersize=8, label='Ternary')
ax_e.plot(seps, b_accs, 's--', color='#348ABD', linewidth=2, markersize=7, label='Binary')
ax_e.plot(seps, bayes_accs, '^:', color='#2ECC71', linewidth=2, markersize=7, label='Bayes Optimal')
ax_e.set_xlabel("Class Separation", fontsize=12)
ax_e.set_ylabel("Test Accuracy (%)", fontsize=12)
ax_e.set_title("(e) Accuracy vs Separation", fontsize=13, fontweight='bold')
ax_e.legend(loc='lower right', fontsize=10)

# --- Row 2, Panel (f): UNKNOWN% vs Bayes uncertainty by region ---
ax_f = fig.add_subplot(gs_fig[1, 2])
# For each separation, compute mean Bayes entropy on test set
mean_bayes_H = []
for sr in sweep_results:
    sep = sr["sep"]
    X_te_s = DATASETS_TEST.get("gaussians", (None, None))[0]  # Use default gaussians test set
    # Re-generate for each separation
    X_s, y_s = make_gaussians(n_samples=2000, separation=sep, random_state=SEED)
    _, _, X_te_s, _ = split_train_test(X_s, y_s)
    p1_te = bayes_posterior_gaussians(X_te_s, sep, sigma=0.5)
    mean_H = np.mean(bayes_entropy(p1_te))
    mean_bayes_H.append(mean_H)

ax_f.scatter(mean_bayes_H, unks, s=100, c=seps, cmap='viridis',
             edgecolors='black', linewidths=1, zorder=3)
for i, sep in enumerate(seps):
    ax_f.annotate(f"$\\Delta$={sep:.1f}", (mean_bayes_H[i], unks[i]),
                  textcoords="offset points", xytext=(8, 5), fontsize=9)
# Fit line
if len(mean_bayes_H) > 2:
    z = np.polyfit(mean_bayes_H, unks, 1)
    x_fit = np.linspace(min(mean_bayes_H), max(mean_bayes_H), 50)
    ax_f.plot(x_fit, np.polyval(z, x_fit), '--', color='gray', alpha=0.7)
    r_corr2, _ = pearsonr(mean_bayes_H, unks)
    ax_f.text(0.95, 0.05, f"r = {r_corr2:.2f}", transform=ax_f.transAxes,
             va='bottom', ha='right', fontsize=11,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
ax_f.set_xlabel("Mean Bayes Entropy $\\langle H(Y|x) \\rangle$", fontsize=12)
ax_f.set_ylabel("UNKNOWN Output Fraction (%)", fontsize=12)
ax_f.set_title("(f) UNKNOWN vs Bayes Uncertainty", fontsize=13, fontweight='bold')

plt.savefig("plots/FINAL_unknown_vs_uncertainty.svg", bbox_inches="tight")
plt.savefig("plots/FINAL_unknown_vs_uncertainty.pdf", bbox_inches="tight")
plt.show()
print("Saved: plots/FINAL_unknown_vs_uncertainty.svg, .pdf")

## Figure 4: Asymmetric Thresholding

In [ ]:
# === CELL 15: Train at 4 delta_fractions on moons ===

DELTA_FRACTIONS = [1.0, 0.50, 0.25, 0.0]
ASYM_RESULTS = {}

for df in DELTA_FRACTIONS:
    print(f"\nTraining with delta_fraction = {df:.2f}")
    asym_pipe = AsymmetricTernaryPipeline(resolution=4, delta_fraction=df)

    result = train_ternary_gs(
        "moons", LAYER_WIDTHS, NPC,
        resolution=4, lr=0.003, lambda_max=0.1, lambda_gamma=2.0,
        steps=5000, batch_size=64,
        pipeline=asym_pipe,
    )

    ASYM_RESULTS[df] = result
    print(f"  Test acc: {result['test_acc']:.3f}  UNK: {result['unknown_frac']:.3f}  "
          f"Gap: {result['gap']:.4f}")

print("\nAsymmetric sweep complete.")

In [ ]:
# === CELL 16: Figure 4 -- 1x4 decision boundaries + accuracy table ===

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

X_all_m = np.vstack([DATASETS["moons"][0], DATASETS_TEST["moons"][0]])
y_all_m = np.hstack([DATASETS["moons"][1], DATASETS_TEST["moons"][1]])

for idx, df in enumerate(DELTA_FRACTIONS):
    ax = axes[idx]
    res = ASYM_RESULTS[df]

    # Compute decision grid
    xx, yy, pred_grid, margin_grid, unk_grid = make_decision_grid(
        X_all_m, res["pipeline"], res["circuit"],
        res["k"], res["npc"], grid_res=200,
    )

    # Decision boundary
    ax.contourf(xx, yy, pred_grid, levels=[-0.5, 0.5, 1.5],
                colors=['#a6c8e0', '#f5c4a1'], alpha=0.6)
    ax.contour(xx, yy, pred_grid, levels=[0.5], colors='black', linewidths=1.5)

    # UNKNOWN overlay
    if unk_grid.max() > 0.001:
        ax.imshow(unk_grid, extent=[xx.min(), xx.max(), yy.min(), yy.max()],
                  origin='lower', cmap='Oranges', alpha=0.6,
                  vmin=0, vmax=max(unk_grid.max(), 0.01), aspect='auto')

    # Data points
    colors = ['#3274A1', '#E1812C']
    for c in [0, 1]:
        mask = y_all_m == c
        ax.scatter(X_all_m[mask, 0], X_all_m[mask, 1], c=colors[c],
                   s=4, alpha=0.3, rasterized=True)

    acc = res["test_acc"]
    unk = res["unknown_frac"]
    ax.set_title(f"$\\delta$ = {df:.2f}\nAcc: {acc:.1%}, UNK: {unk:.1%}",
                 fontsize=13, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig("plots/FINAL_asymmetric_thresholding.svg", bbox_inches="tight")
plt.savefig("plots/FINAL_asymmetric_thresholding.pdf", bbox_inches="tight")
plt.show()

# Print accuracy table
print(f"\n{'delta':>8s} | {'Test Acc':>10s} | {'UNK%':>8s} | {'Gap':>8s}")
print("-" * 45)
for df in DELTA_FRACTIONS:
    r = ASYM_RESULTS[df]
    print(f"{df:8.2f} | {r['test_acc']:10.3f} | {r['unknown_frac']*100:7.1f}% | {r['gap']:8.4f}")

print("\nSaved: plots/FINAL_asymmetric_thresholding.svg, .pdf")

## Figure 5: Encoding Resolution

In [ ]:
# === CELL 18: Train at resolutions [2, 4, 8, 16] ===

RESOLUTIONS = [2, 4, 8, 16]
# Scale body widths with resolution to keep neuron-to-input ratio comparable
RES_BODY_WIDTHS = {
    2:  [128, 128, 128],
    4:  [256, 256, 256],
    8:  [512, 512, 512],
    16: [1024, 1024, 1024],
}

RES_RESULTS = {}

for res in RESOLUTIONS:
    body = RES_BODY_WIDTHS[res]
    lw = body + [NPC * 2]  # output = npc * k = 100 * 2 = 200
    K = res - 1  # number of thresholds from uniform_thresholds(res)
    input_dim = 2 * K  # 2D data

    print(f"\nResolution {res}: K={K}, input_dim={input_dim}, body={body}, output={NPC*2}")

    t_res = train_ternary_gs(
        "moons", lw, NPC,
        resolution=res, lr=0.003, lambda_max=0.1, lambda_gamma=2.0,
        steps=5000, batch_size=64,
    )

    RES_RESULTS[res] = t_res
    print(f"  Test acc: {t_res['test_acc']:.3f}  UNK: {t_res['test_unk']:.3f}")

print("\nResolution sweep complete.")

In [ ]:
# === CELL 19: Figure 5 -- dual-axis plot + decision boundaries ===

fig = plt.figure(figsize=(22, 5))
gs_fig = GridSpec(1, 5, figure=fig, width_ratios=[1.5, 1, 1, 1, 1], wspace=0.3)

# Left panel: dual-axis line plot
ax_left = fig.add_subplot(gs_fig[0, 0])
res_vals = list(RES_RESULTS.keys())
accs = [RES_RESULTS[r]["test_acc"] * 100 for r in res_vals]
unks = [RES_RESULTS[r]["test_unk"] * 100 for r in res_vals]

color_acc = '#E24A33'
color_unk = '#348ABD'

ax_left.plot(res_vals, accs, 'o-', color=color_acc, linewidth=2.5, markersize=10, label='Accuracy')
ax_left.set_xlabel("Resolution", fontsize=12)
ax_left.set_ylabel("Test Accuracy (%)", fontsize=12, color=color_acc)
ax_left.tick_params(axis='y', labelcolor=color_acc)
ax_left.set_xticks(res_vals)

ax_right = ax_left.twinx()
ax_right.plot(res_vals, unks, 's--', color=color_unk, linewidth=2, markersize=8, label='UNKNOWN %')
ax_right.set_ylabel("UNKNOWN Output Fraction (%)", fontsize=12, color=color_unk)
ax_right.tick_params(axis='y', labelcolor=color_unk)

# Combined legend
lines1, labels1 = ax_left.get_legend_handles_labels()
lines2, labels2 = ax_right.get_legend_handles_labels()
ax_left.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=10)
ax_left.set_title("Resolution Trade-off", fontsize=13, fontweight='bold')

# Right 4 panels: decision boundaries at each resolution
X_all_m = np.vstack([DATASETS["moons"][0], DATASETS_TEST["moons"][0]])
y_all_m = np.hstack([DATASETS["moons"][1], DATASETS_TEST["moons"][1]])

for i, res in enumerate(res_vals):
    ax = fig.add_subplot(gs_fig[0, i + 1])
    r = RES_RESULTS[res]
    xx, yy, pg, mg, ug = make_decision_grid(
        X_all_m, r["pipeline"], r["circuit"],
        r["k"], r["npc"], grid_res=150,
    )
    ax.contourf(xx, yy, pg, levels=[-0.5, 0.5, 1.5],
                colors=['#a6c8e0', '#f5c4a1'], alpha=0.6)
    ax.contour(xx, yy, pg, levels=[0.5], colors='black', linewidths=1)
    if ug.max() > 0.001:
        ax.imshow(ug, extent=[xx.min(), xx.max(), yy.min(), yy.max()],
                  origin='lower', cmap='Oranges', alpha=0.5,
                  vmin=0, vmax=max(ug.max(), 0.01), aspect='auto')
    for c in [0, 1]:
        mask = y_all_m == c
        ax.scatter(X_all_m[mask, 0], X_all_m[mask, 1],
                   c=['#3274A1', '#E1812C'][c], s=3, alpha=0.2, rasterized=True)
    K = res - 1
    ax.set_title(f"Res={res} (K={K})\nAcc: {r['test_acc']:.1%}",
                 fontsize=11, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

plt.savefig("plots/FINAL_resolution_vs_accuracy.svg", bbox_inches="tight")
plt.savefig("plots/FINAL_resolution_vs_accuracy.pdf", bbox_inches="tight")
plt.show()
print("Saved: plots/FINAL_resolution_vs_accuracy.svg, .pdf")

## Figure 6: Hardening Gap

In [ ]:
# === CELL 21: Figure 6 -- grouped bar chart ===

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(ALL_DATASET_NAMES))
width = 0.35

binary_gaps = [abs(RESULTS[ds]["binary"]["gap"]) * 100 for ds in ALL_DATASET_NAMES]
ternary_gaps = [abs(RESULTS[ds]["ternary"]["gap"]) * 100 for ds in ALL_DATASET_NAMES]

bars_b = ax.bar(x - width/2, binary_gaps, width, label='Binary DLGN',
                color='#348ABD', alpha=0.8, edgecolor='white')
bars_t = ax.bar(x + width/2, ternary_gaps, width, label='Ternary PST-DTLGN',
                color='#E24A33', alpha=0.8, edgecolor='white')

# Annotate values on bars
for bar in bars_b:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.05,
            f'{h:.2f}%', ha='center', va='bottom', fontsize=9)
for bar in bars_t:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.05,
            f'{h:.2f}%', ha='center', va='bottom', fontsize=9)

# Zero-gap reference line
ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(1.0, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
ax.text(len(ALL_DATASET_NAMES) - 0.5, 1.05, '1% threshold', fontsize=9,
        color='gray', ha='right')

ax.set_xlabel('Dataset', fontsize=12)
ax.set_ylabel('Hardening Gap (|soft acc - hard acc|, %)', fontsize=12)
ax.set_title('Hardening Gap: Binary DLGN vs Ternary PST-DTLGN', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([DATASET_LABELS[ds] for ds in ALL_DATASET_NAMES], fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("plots/FINAL_hardening_gap.svg", bbox_inches="tight")
plt.savefig("plots/FINAL_hardening_gap.pdf", bbox_inches="tight")
plt.show()
print("Saved: plots/FINAL_hardening_gap.svg, .pdf")

## Summary Tables

In [ ]:
# === CELL 23: Table 1 -- Head-to-head comparison ===

print("Table 1: Head-to-Head Comparison (Test Set)")
print("=" * 100)
print(f"{'Dataset':15s} | {'Bin Acc':>8s} | {'Tern Acc':>9s} | {'Delta':>6s} | "
      f"{'UNK%':>6s} | {'Gap(Bin)':>9s} | {'Gap(Tern)':>10s}")
print("-" * 100)

for ds in ALL_DATASET_NAMES:
    t = RESULTS[ds]["ternary"]
    b = RESULTS[ds]["binary"]
    delta = t["test_acc"] - b["test_acc"]
    print(f"{ds:15s} | {b['test_acc']:7.1%} | {t['test_acc']:8.1%} | "
          f"{delta:+5.1%} | {t['test_unk']*100:5.1f}% | "
          f"{abs(b['gap'])*100:8.2f}% | {abs(t['gap'])*100:9.2f}%")

# LaTeX version
print("\n\n% LaTeX table:")
print(r"\begin{tabular}{lcccccc}")
print(r"\toprule")
print(r"Dataset & Bin Acc & Tern Acc & $\Delta$ & UNK\% & Gap(Bin) & Gap(Tern) \\")
print(r"\midrule")
for ds in ALL_DATASET_NAMES:
    t = RESULTS[ds]["ternary"]
    b = RESULTS[ds]["binary"]
    delta = t["test_acc"] - b["test_acc"]
    print(f"{DATASET_LABELS[ds]} & {b['test_acc']:.1%} & {t['test_acc']:.1%} & "
          f"{delta:+.1%} & {t['test_unk']*100:.1f}\\% & "
          f"{abs(b['gap'])*100:.2f}\\% & {abs(t['gap'])*100:.2f}\\% \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")

In [ ]:
# === CELL 24: Table 2 -- Abstention quality (ternary only) ===

print("Table 2: Abstention Quality (Ternary PST-DTLGN, Test Set)")
print("=" * 90)
print(f"{'Dataset':15s} | {'AUC':>8s} | {'Acc@90%':>8s} | {'Acc@50%':>8s} | "
      f"{'Median Margin':>14s} | {'UNK%':>6s}")
print("-" * 90)

for ds in ALL_DATASET_NAMES:
    t = RESULTS[ds]["ternary"]
    X_te, y_te = DATASETS_TEST[ds]

    preds = t["test_preds"]
    margins = t["test_margins"]
    covs, accs, _ = accuracy_vs_coverage(preds, y_te, margins)

    # AUC
    valid = ~np.isnan(accs)
    auc = np.trapezoid(accs[valid], covs[valid]) if np.any(valid) else 0.0

    # Accuracy at specific coverage levels
    def acc_at_coverage(covs, accs, target):
        idx = np.argmin(np.abs(covs - target))
        return accs[idx] if not np.isnan(accs[idx]) else float('nan')

    acc_90 = acc_at_coverage(covs, accs, 0.9)
    acc_50 = acc_at_coverage(covs, accs, 0.5)
    median_margin = np.median(margins)

    print(f"{ds:15s} | {auc:8.3f} | {acc_90:7.1%} | {acc_50:7.1%} | "
          f"{median_margin:14.1f} | {t['test_unk']*100:5.1f}%")

In [ ]:
# === CELL 25: Table 3 -- Gate diversity (both architectures) ===

print("Table 3: Gate Diversity")
print("=" * 110)
print(f"{'Dataset':15s} | {'Tern Unique':>12s} {'Tern EffDiv':>12s} {'Tern Redund':>12s} | "
      f"{'Bin Unique':>11s} {'Bin EffDiv':>11s} {'Bin Redund':>11s}")
print("-" * 110)

for ds in ALL_DATASET_NAMES:
    t_hr = RESULTS[ds]["ternary"]["harden_result"]
    b_hr = RESULTS[ds]["binary"]["harden_result"]

    t_div = gate_diversity(t_hr)
    b_div = gate_diversity(b_hr)
    t_red = functional_redundancy(t_hr)
    b_red = functional_redundancy(b_hr)

    print(f"{ds:15s} | "
          f"{t_div['unique_gates']:12d} {t_div['effective_diversity']:12.1f} "
          f"{t_red['redundancy_ratio']:11.1%} | "
          f"{b_div['unique_gates']:11d} {b_div['effective_diversity']:11.1f} "
          f"{b_red['redundancy_ratio']:10.1%}")

In [ ]:
# === CELL 27: Print all saved figure paths ===

import glob

final_plots = sorted(glob.glob("plots/FINAL_*.svg") + glob.glob("plots/FINAL_*.pdf"))
print("Generated figures:")
print("=" * 60)
for p in final_plots:
    size_kb = os.path.getsize(p) / 1024
    print(f"  {p:45s}  ({size_kb:.0f} KB)")
print(f"\nTotal: {len(final_plots)} files")